# Streaming MLP Forward Return

Build LOB features from MBO data, add a forward-return target, then run rolling time-series validation with the streaming `TorchAdapter` and a configurable PyTorch MLP.

In [1]:
# Source - https://stackoverflow.com/a/10472712
# Posted by Andrew_1510, modified by community. See post 'Timeline' for change history
# Retrieved 2026-06-28, License - CC BY-SA 4.0

%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations

import sys
from collections.abc import Sequence
from pathlib import Path

import numpy as np
import polars as pl
import torch
from matplotlib import pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tools.data import FeatureLoader, expand_dates
from tools.features import LOBFeatures, depth_meta
from tools.filters import intraday_time, level_taken, tight_spread, trade_size
from tools.model import TorchAdapter
from tools.pipeline import Pipeline
from tools.score import get_unit_pnl, rmse
from tools.transform import Standardizer

In [3]:
expand_dates("20260501-20260507")

['2026-05-01', '2026-05-04', '2026-05-05', '2026-05-06', '2026-05-07']

In [4]:
def divide_dates(*args):
    dates = []
    for i in range(1, len(args)):
        dates.append(
            expand_dates(
                f"{args[i - 1]}-{args[i]}",
                end_date=False if i < len(args) - 1 else True,
            )
        )
    return dates

In [5]:
# Data
PROD = "ES"
# ROLLING_DATES = [["2026-05-01", "2026-05-04", "2026-05-05"], ["2026-05-06"], ["2026-05-07"]]
ROLLING_DATES = divide_dates(20260323, 20260410, 20260425, 20260510, 20260524)
TEST_DATES = expand_dates('20260525-20260529')
L2_DEPTH = 5
MODEL_BATCH_SIZE = 8_192
POLARS_ENGINE = "streaming"
ORDERBOOK_PATH = str(ROOT / f"data/orderbook_l2_parquet/{{prod}}M6_{{d}}_{{tag}}_{{prod_s}}_full_day_l2_d{L2_DEPTH}.parquet")
RAW_CONTEXT_COLS = ("ts_event", "instrument_id")
# RAW_CONTEXT_COLS = ("ts_event", "ts_recv", "symbol")
REGULAR_HOURS_START = "09:30"
REGULAR_HOURS_END = "16:00"
REGULAR_HOURS_TZ = "America/New_York"

# Feature knobs
IMBALANCE_DEPTHS = [1, 3, 5]
IMBALANCE_LOG = True
TRADE_MOMENTUM_LOG = True
WEIGHTED_PRICE_DEPTH = 5
WEIGHTED_PRICE_SIZES = [2, 5, 10]
TRADE_MOMENTUM_HALF_LIVES = [1, 10, 30, 120]  # seconds

# Forward-return target knobs
FUTURE_HORIZONS = ["10s", "30s", "60s"]
FUTURE_WEIGHTS = [0.5, 0.3, 0.2]
TARGET = "forward_mid_return_bps"
TEST_PNL_THRESHOLD = 0.2

# MLP/search knobs
SEED = 7
SAMPLER = "random"
N_TRIALS = 20
EPOCHS = 5
DEVICE = "cuda"

DEFAULT_MLP_PARAMS = {
    "hidden_layers": 2,
    "hidden_units": [128, 64],
    "activation": "silu",
    "dropout": 0.05,
    "lr": 1e-3,
    "weight_decay": 1e-4,
}
TUNE_ARCHITECTURE = True
HIDDEN_LAYER_CHOICES = [1, 2, 3]
HIDDEN_UNITS_CHOICES = [32, 64, 128, 256]
ACTIVATION_CHOICES = ["silu", "relu", "gelu"]
DROPOUT_CHOICES = [0.0, 0.05, 0.1]
LEARNING_RATE_RANGE = (1e-4, 3e-3)
WEIGHT_DECAY_RANGE = (1e-6, 1e-2)

UNDEF_PRICE = 9_223_372_036_854_775_807
assert L2_DEPTH >= max([*IMBALANCE_DEPTHS, WEIGHTED_PRICE_DEPTH])
TICKSIZE = 250000000

np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("DEVICE='cuda' requested, but torch.cuda.is_available() is False.")

In [6]:
def tag(x: object) -> str:
    return str(x).replace(".", "p")


def feature_exprs() -> dict[str, pl.Expr]:
    exprs = {}
    for depth in IMBALANCE_DEPTHS:
        exprs[f"imb_d{depth}"] = LOBFeatures.book_imbalance(depth, log=IMBALANCE_LOG)
    for size in sorted(set([*WEIGHTED_PRICE_SIZES])):
        exprs[f"weighted_price_sz{tag(size)}"] = LOBFeatures.size_weighted_avg_price(
            WEIGHTED_PRICE_DEPTH,
            size,
        )
    return exprs


def stateful_features() -> tuple[object, ...]:
    features = []
    for half_life in TRADE_MOMENTUM_HALF_LIVES:
        suffix = f"{tag(half_life)}s"
        features.append(
            LOBFeatures.TradeMomentum(
                f"trade_momentum_hl{suffix}",
                half_life,
                log=TRADE_MOMENTUM_LOG,
                normalized=False,
            )
        )
        features.append(
            LOBFeatures.PushMomentum(
                f"push_momentum_hl{suffix}",
                half_life,
                log=TRADE_MOMENTUM_LOG,
                normalized=False,
            )
        )
        features.append(
            LOBFeatures.PullMomentum(
                f"pull_momentum_hl{suffix}",
                half_life,
                log=TRADE_MOMENTUM_LOG,
                normalized=False,
            )
        )
        for mode in ("side", "volume"):
            features.append(
                LOBFeatures.TradeCorrelation(
                    f"trade_corr_{mode}_hl{suffix}",
                    half_life,
                    mode=mode,
                )
            )
        features.append(LOBFeatures.LogReturn(f"log_return_hl{suffix}", half_life))
        features.append(LOBFeatures.EwmaSpread(f"ewma_spread_hl{suffix}", half_life))
    return tuple(features)


FEATURE_EXPRS = feature_exprs()
STATEFUL_FEATURES = stateful_features()
FEATURES = [*FEATURE_EXPRS, *[feature.name for feature in STATEFUL_FEATURES]]
# FEATURES = ["imb_d1"]
# FEATURES = ["weighted_price_sz5"]

MID = (pl.col("bid_px_0") + pl.col("ask_px_0")) / 2
# META_COLS = ["ts_event", "ts_recv", "symbol", *depth_meta(L2_DEPTH)]
META_COLS = ["ts_event", "instrument_id", *depth_meta(L2_DEPTH)]
VALID_ROWS = (
    (pl.col("bid_px_0") != UNDEF_PRICE)
    & (pl.col("ask_px_0") != UNDEF_PRICE)
    & (pl.col("ask_px_0") > pl.col("bid_px_0"))
    & pl.col(TARGET).is_not_null()
    & pl.all_horizontal([pl.col(c).is_finite() for c in FEATURES])
)
REGULAR_HOURS = intraday_time(REGULAR_HOURS_START, REGULAR_HOURS_END, timezone=REGULAR_HOURS_TZ)
TIGHT_SPREAD = tight_spread(TICKSIZE)
VALID_REGULAR_ROWS = VALID_ROWS & REGULAR_HOURS & TIGHT_SPREAD
TRAIN_ROWS = VALID_REGULAR_ROWS & (level_taken() | trade_size(0.3))
FEATURES

['imb_d1',
 'imb_d3',
 'imb_d5',
 'weighted_price_sz2',
 'weighted_price_sz5',
 'weighted_price_sz10',
 'trade_momentum_hl1s',
 'trade_momentum_hl10s',
 'trade_momentum_hl30s',
 'trade_momentum_hl120s']

In [7]:
REGULAR_HOURS

<Expr ['[([(col("ts_event").dt.convert…'] at 0x77420D99FCE0>

In [8]:
loader = FeatureLoader(
    prod=PROD,
    feature_exprs=FEATURE_EXPRS,
    stateful_features=STATEFUL_FEATURES,
    return_exprs={TARGET: MID},
    horizons=FUTURE_HORIZONS,
    weights=FUTURE_WEIGHTS,
    l2_depth=None,
    path=ORDERBOOK_PATH,
    return_time="ts_event",
    return_by=("publisher_id", "instrument_id"),
)


In [ ]:
from tools.data import DataSource

FEATURE_TEST_SCORE = get_unit_pnl(TEST_PNL_THRESHOLD)
FEATURE_TEST_SCORE_DESCENDING = True

test_date_src = DataSource(
    dates=TEST_DATES,
    loader=loader,
    target=TARGET,
    features=FEATURES,
    filters=(VALID_REGULAR_ROWS,),
    polars_engine=POLARS_ENGINE,
)

feature_test_states = dict.fromkeys(FEATURES)
feature_test_rows = 0
for x, y_true, ctx in test_date_src.batches(MODEL_BATCH_SIZE):
    feature_test_rows += int(ctx["n"])
    for idx, feature in enumerate(FEATURES):
        feature_test_states[feature] = FEATURE_TEST_SCORE(
            y_true,
            x[:, idx],
            ctx,
            combine_with=feature_test_states[feature],
        )

feature_test_scores = (
    pl.DataFrame(
        [
            {
                "feature": feature,
                "score": getattr(FEATURE_TEST_SCORE, "__name__", "score"),
                "test_score": float(state),
                "score_n": int(getattr(state, "n", 0)),
                "rows": feature_test_rows,
            }
            for feature, state in feature_test_states.items()
            if state is not None
        ]
    )
    .sort("test_score", descending=FEATURE_TEST_SCORE_DESCENDING)
)

feature_test_scores

Loading data: 2.30Mrow [05:11, 12.7krow/s]

The architecture is controlled by `hidden_layers` and either one `hidden_units` value, a `hidden_units` list, or per-layer `hidden_units_l1`, `hidden_units_l2`, ... values. Set `TUNE_ARCHITECTURE = False` and `N_TRIALS = 1` to train only `DEFAULT_MLP_PARAMS`.

In [ ]:
def activation_layer(name: str) -> torch.nn.Module:
    name = name.lower()
    if name == "relu":
        return torch.nn.ReLU()
    if name == "gelu":
        return torch.nn.GELU()
    if name == "silu":
        return torch.nn.SiLU()
    if name == "tanh":
        return torch.nn.Tanh()
    raise ValueError(f"unsupported activation: {name}")


def hidden_sizes_from_params(params: dict[str, object]) -> list[int]:
    hidden_layers = int(params.get("hidden_layers", DEFAULT_MLP_PARAMS["hidden_layers"]))
    if hidden_layers < 0:
        raise ValueError("hidden_layers must be non-negative")

    units = params.get("hidden_units")
    if isinstance(units, str):
        sizes = [int(part.strip()) for part in units.split(",") if part.strip()]
    elif isinstance(units, Sequence):
        sizes = [int(unit) for unit in units]
    elif units is not None:
        sizes = [int(units)] * hidden_layers
    else:
        default_units = DEFAULT_MLP_PARAMS["hidden_units"]
        fallback = default_units[0] if isinstance(default_units, Sequence) else default_units
        sizes = [int(params.get(f"hidden_units_l{i + 1}", fallback)) for i in range(hidden_layers)]

    if len(sizes) < hidden_layers:
        fill = sizes[-1] if sizes else int(DEFAULT_MLP_PARAMS["hidden_units"][0])
        sizes.extend([fill] * (hidden_layers - len(sizes)))
    sizes = sizes[:hidden_layers]
    if any(width <= 0 for width in sizes):
        raise ValueError(f"hidden layer widths must be positive: {sizes}")
    return sizes


def build_mlp(params: dict[str, object]) -> torch.nn.Module:
    torch.manual_seed(int(params.get("seed", SEED)))
    hidden_sizes = hidden_sizes_from_params(params)
    activation = str(params.get("activation", DEFAULT_MLP_PARAMS["activation"]))
    dropout = float(params.get("dropout", DEFAULT_MLP_PARAMS["dropout"]))

    layers: list[torch.nn.Module] = []
    in_features = len(FEATURES)
    for width in hidden_sizes:
        layers.append(torch.nn.Linear(in_features, width))
        layers.append(activation_layer(activation))
        if dropout > 0.0:
            layers.append(torch.nn.Dropout(dropout))
        in_features = width
    layers.append(torch.nn.Linear(in_features, 1))

    model = torch.nn.Sequential(*layers)
    setattr(model, "_hidden_sizes", hidden_sizes)
    return model


def build_optimizer(parameters, params: dict[str, object]):
    return torch.optim.AdamW(
        parameters,
        lr=float(params.get("lr", DEFAULT_MLP_PARAMS["lr"])),
        weight_decay=float(params.get("weight_decay", DEFAULT_MLP_PARAMS["weight_decay"])),
    )


def mlp_search_space(trial) -> dict[str, object]:
    if not TUNE_ARCHITECTURE:
        return dict(DEFAULT_MLP_PARAMS)

    hidden_layers = trial.suggest_categorical("hidden_layers", HIDDEN_LAYER_CHOICES)
    params: dict[str, object] = {
        "hidden_layers": hidden_layers,
        "activation": trial.suggest_categorical("activation", ACTIVATION_CHOICES),
        "dropout": trial.suggest_categorical("dropout", DROPOUT_CHOICES),
        "lr": trial.suggest_float("lr", *LEARNING_RATE_RANGE, log=True),
        "weight_decay": trial.suggest_float("weight_decay", *WEIGHT_DECAY_RANGE, log=True),
        "seed": SEED,
    }
    for layer_idx in range(1, int(hidden_layers) + 1):
        params[f"hidden_units_l{layer_idx}"] = trial.suggest_categorical(
            f"hidden_units_l{layer_idx}",
            HIDDEN_UNITS_CHOICES,
        )
    return params


hidden_sizes_from_params(DEFAULT_MLP_PARAMS)

In [ ]:
pipeline = Pipeline(
    rolling_dates=ROLLING_DATES,
    test_dates=TEST_DATES,
    adapter=TorchAdapter(
        module_builder=build_mlp,
        optimizer_builder=build_optimizer,
        epochs=EPOCHS,
        batch_size=MODEL_BATCH_SIZE,
        device=DEVICE,
        streaming=True,
    ),
    target=TARGET,
    features=FEATURES,
    data_loader=loader,
    search_space=mlp_search_space,
    val_score=rmse,
    transform=Standardizer(FEATURES),
    train_filters=(TRAIN_ROWS,),
    val_filters=(VALID_REGULAR_ROWS,),
    test_filters=(VALID_REGULAR_ROWS,),
    sampler=SAMPLER,
    n_trials=N_TRIALS,
    cache_arrays=False,
    seed=SEED,
    score_direction="minimize",
    polars_engine=POLARS_ENGINE,
)
pipeline

In [ ]:
ROLLING_DATES[-1][:1]

In [ ]:
train_result = pipeline.train(verbose=2)
train_result

In [ ]:
model = pipeline.get_model()
best_hidden_sizes = getattr(model.module if hasattr(model, "module") else model, "_hidden_sizes", None)
n_params = sum(param.numel() for param in model.parameters())
best_hidden_sizes, n_params, pipeline.best_params

In [ ]:
rmse_result = pipeline.test(rmse)
rmse_result

In [ ]:
pnl_result = pipeline.test(get_unit_pnl(0.0))
pnl_result

In [ ]:
pnl_threshold_result = pipeline.test(get_unit_pnl(TEST_PNL_THRESHOLD))
pnl_threshold_result

In [ ]:
print(np.mean(pnl_threshold_result["y_pred"]), np.std(pnl_threshold_result["y_pred"]))
_ = plt.hist(pnl_threshold_result["y_pred"], bins=100, log=True, density=False)
plt.xlabel("prediction")
plt.ylabel("count")

In [ ]:
test_src = DataSource(
    dates=TEST_DATES,
    loader=loader,
    target=TARGET,
    features=FEATURES,
    filters=(VALID_REGULAR_ROWS,),
    polars_engine=POLARS_ENGINE,
)
y_true, _ = test_src.labels()
pred_eval = pl.DataFrame({"y_true": y_true, "y_pred": pnl_threshold_result["y_pred"]})
n_pred = pred_eval.height
pred_eval = pred_eval.with_columns(
    (((pl.col("y_pred").rank("average") - 1) * 10 / n_pred).floor().clip(0, 9).cast(pl.Int8)).alias("pred_decile")
)

pred_eval.group_by("pred_decile").agg(
    pl.len().alias("n"),
    pl.col("y_pred").mean().alias("mean_pred"),
    pl.col("y_true").mean().alias("mean_forward_return_bps"),
    (pl.col("y_true") * pl.col("y_pred").sign()).mean().alias("mean_signed_return_bps"),
).sort("pred_decile")